# 3.benchmark_analysis_15test.py

Autogenerated from the matching source file in `14.15Test/code_src`.

In [1]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ruptures as rpt
from matplotlib.patches import Patch
from sklearn.metrics import auc, f1_score, precision_score, recall_score, roc_curve

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

ROOT = Path("/Users/jane/Documents/202511吾-Systems/14.15Test")
MERGED_DIR = ROOT / "merged"
TABLE_DIR = ROOT / "tables"
PLOT_DIR = ROOT / "plots"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

events = pd.read_csv(MERGED_DIR / "crypto_market_events.csv", parse_dates=["event_date"])["event_date"]
df = pd.read_csv(
    MERGED_DIR / "TDA_SP500_VIX_BTC_BVOL_CVI_15_merged_labeled_with_network.csv",
    parse_dates=["date"],
)
df = df.sort_values("date").reset_index(drop=True)

df["event_label"] = 0
for e in events:
    df.loc[
        (df["date"] >= e - pd.Timedelta(days=21))
        & (df["date"] <= e + pd.Timedelta(days=21)),
        "event_label",
    ] = 1

for col in ["SP500", "VIX", "BVOL", "CVI", "vol_btc_parkinson", "vol_btc_rogers", "fng_value"]:
    if col in df.columns:
        df[col] = df[col].ffill().bfill()

df["sp_ret"] = df["SP500"].pct_change()
df["sp_vol7"] = df["sp_ret"].rolling(7).std()
df["fear&greed"] = df["fng_value"]
df["fear_greed"] = df["fng_value"]
df = df.ffill().bfill()

tda_indicators = [
    "vol_wass_betti0", "vol_wass_betti1", "vol_wass_entropy",
    "log_wass_betti0", "log_wass_betti1", "log_wass_entropy",
    "log_dtw_betti0", "log_dtw_betti1", "log_dtw_entropy",
    "vol_dtw_betti0", "vol_dtw_betti1", "vol_dtw_entropy",
]
market_indicators = ["SP500", "sp_ret", "sp_vol7", "VIX", "fear&greed", "BVOL", "CVI"]
btc_indicators = ["vol_btc_parkinson", "vol_btc_rogers"]
network_indicators = [
    "vol_wass_clustering", "vol_wass_degree_centrality", "vol_wass_pagerank",
    "log_wass_clustering", "log_wass_degree_centrality", "log_wass_pagerank",
    "log_dtw_clustering", "log_dtw_degree_centrality", "log_dtw_pagerank",
    "vol_dtw_clustering", "vol_dtw_degree_centrality", "vol_dtw_pagerank",
]
all_indicators = tda_indicators + market_indicators + btc_indicators + network_indicators

color_tda = "#4DBBD5"
color_market = "#3C5488"
color_btc = "#E64B35"
color_network = "#999999"

auc_rows = []
y_true = df["event_label"].to_numpy()
for col in all_indicators:
    y_score = df[col].to_numpy(dtype=float)
    mask = np.isfinite(y_score)
    fpr, tpr, _ = roc_curve(y_true[mask], y_score[mask])
    roc_auc = auc(fpr, tpr)
    if col in tda_indicators:
        group = "TDA"
    elif col in market_indicators:
        group = "Market & sentiment"
    elif col in btc_indicators:
        group = "BTC volatility"
    else:
        group = "Network baseline"
    auc_rows.append({"indicator": col, "auc": roc_auc, "group": group})

auc_df = pd.DataFrame(auc_rows).sort_values("auc", ascending=False)
auc_df.to_csv(TABLE_DIR / "benchmark_auc_results_15.csv", index=False)


def get_group_color(name):
    if name in tda_indicators:
        return color_tda
    if name in market_indicators:
        return color_market
    if name in btc_indicators:
        return color_btc
    return color_network


plot_df = auc_df.copy()
colors = [get_group_color(name) for name in plot_df["indicator"]]
y_pos = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(15, 9))
bars = ax.barh(y_pos, plot_df["auc"], color=colors, edgecolor="black", linewidth=0.7, alpha=0.9)
ax.set_yticks(y_pos)
ax.set_yticklabels([str(x).replace("_", " ") for x in plot_df["indicator"]], fontsize=11)
ax.invert_yaxis()
ax.set_xlim(0, max(0.8, plot_df["auc"].max() + 0.06))
ax.set_xlabel("AUC", fontsize=14)
ax.set_title(
    "ROC–AUC Benchmark Results across TDA, market, BTC and network indicators (15 assets)",
    fontsize=14,
)
ax.xaxis.grid(True, linestyle="--", alpha=0.3)
for bar, value in zip(bars, plot_df["auc"]):
    ax.text(
        bar.get_width() + 0.008,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        va="center",
        ha="left",
        fontsize=11,
    )
legend_handles = [
    Patch(facecolor=color_tda, edgecolor="black", label="Topological features"),
    Patch(facecolor=color_market, edgecolor="black", label="Market & sentiment indicators"),
    Patch(facecolor=color_btc, edgecolor="black", label="BTC volatility"),
    Patch(facecolor=color_network, edgecolor="black", label="Network baseline"),
]
ax.legend(handles=legend_handles, fontsize=12, frameon=False, loc="lower right")
plt.tight_layout()
fig.savefig(PLOT_DIR / "fig_auc_benchmark_15.png", dpi=900, bbox_inches="tight")
fig.savefig(PLOT_DIR / "fig_auc_benchmark_15.pdf", dpi=900, bbox_inches="tight")
plt.close(fig)


def detect_cps(series, method="PELT", n_bkps=10):
    x = series.values.astype(float).reshape(-1, 1)
    n = len(x)
    pen_val = np.log(n)
    if method == "PELT":
        algo = rpt.Pelt(model="rbf").fit(x)
        cps = algo.predict(pen=pen_val)
    elif method == "BinSeg":
        algo = rpt.Binseg(model="rbf").fit(x)
        cps = algo.predict(n_bkps=n_bkps)
    elif method == "CUSUM":
        algo = rpt.KernelCPD(kernel="linear").fit(x)
        cps = algo.predict(n_bkps=n_bkps)
    else:
        raise ValueError(method)
    return cps[:-1]


def compute_f1_from_cps(cp_dates, event_dates, window=21):
    if len(cp_dates) == 0:
        return 0.0, 0.0, 0.0
    all_dates = pd.date_range(start=min(event_dates.min(), min(cp_dates)), end=max(event_dates.max(), max(cp_dates)))
    y_true = pd.Series(0, index=all_dates)
    y_pred = pd.Series(0, index=all_dates)
    for e in event_dates:
        y_true.loc[(all_dates >= e - pd.Timedelta(days=window)) & (all_dates <= e + pd.Timedelta(days=window))] = 1
    for c in cp_dates:
        y_pred.loc[(all_dates >= c - pd.Timedelta(days=window)) & (all_dates <= c + pd.Timedelta(days=window))] = 1
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    return precision, recall, f1


cp_rows = []
for indicator in all_indicators:
    series = df[indicator]
    for method in ["PELT", "CUSUM", "BinSeg"]:
        cps = detect_cps(series, method=method)
        cp_dates = [df["date"].iloc[c] for c in cps if c < len(df)]
        precision, recall, f1 = compute_f1_from_cps(cp_dates, events, window=21)
        cp_rows.append({
            "indicator": indicator,
            "method": method,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "n_changepoints": len(cp_dates),
        })

cp_df = pd.DataFrame(cp_rows).sort_values("f1", ascending=False)
cp_df.to_csv(TABLE_DIR / "benchmark_changepoint_f1_results_15.csv", index=False)

plot_cp = cp_df.sort_values("f1", ascending=True).reset_index(drop=True)
colors = [get_group_color(ind) for ind in plot_cp["indicator"]]
labels = [f"{row['indicator']} ({row['method']})" for _, row in plot_cp.iterrows()]
y = np.arange(len(plot_cp))

fig, ax = plt.subplots(figsize=(24, 14))
bars = ax.barh(y, plot_cp["f1"], color=colors, edgecolor="black", linewidth=0.5)
for i, bar in enumerate(bars):
    ax.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{plot_cp['f1'].iloc[i]:.3f}",
        va="center",
        ha="left",
        fontsize=11,
    )
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel("F1 Score", fontsize=14)
ax.set_xlim(0, plot_cp["f1"].max() + 0.08)
ax.set_title("Structural Break Detection Benchmark (15 assets)", fontsize=15)
ax.xaxis.grid(True, linestyle="--", alpha=0.3)
ax.yaxis.grid(False)
ax.legend(handles=legend_handles, fontsize=12, loc="lower right", frameon=False)
plt.tight_layout()
fig.savefig(PLOT_DIR / "fig_f1_benchmark_15.png", dpi=900, bbox_inches="tight", transparent=True)
fig.savefig(PLOT_DIR / "fig_f1_benchmark_15.pdf", dpi=900, bbox_inches="tight", transparent=True)
plt.close(fig)

manifest = pd.DataFrame([
    {"file": "benchmark_auc_results_15.csv", "path": str(TABLE_DIR / "benchmark_auc_results_15.csv")},
    {"file": "benchmark_changepoint_f1_results_15.csv", "path": str(TABLE_DIR / "benchmark_changepoint_f1_results_15.csv")},
    {"file": "fig_auc_benchmark_15.png", "path": str(PLOT_DIR / "fig_auc_benchmark_15.png")},
    {"file": "fig_f1_benchmark_15.png", "path": str(PLOT_DIR / "fig_f1_benchmark_15.png")},
])
manifest.to_csv(TABLE_DIR / "benchmark_analysis_manifest_15.csv", index=False)

print("Benchmark analysis finished.")
print(auc_df.head())


Benchmark analysis finished.
           indicator       auc             group
9     vol_dtw_betti0  0.664445               TDA
3    log_wass_betti0  0.661094               TDA
6     log_dtw_betti0  0.657995               TDA
32  vol_dtw_pagerank  0.654288  Network baseline
0    vol_wass_betti0  0.652975               TDA
